In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# load data - change the csv name if yours is different
df = pd.read_csv(r"C:\AHAMMED\AI-Powered E-commerce Customer Intelligence System\Dataset\data.csv", encoding='unicode_escape')

# quick check on what we are dealing with
print("Raw data shape:", df.shape)
# print(df.isnull().sum()) # just checking how many nulls we have

# drop missing customer IDs[cite: 1] - can't cluster people we don't know
df.dropna(subset=['CustomerID'], inplace=True)

# ditch the cancelled orders (InvoiceNo starts with 'C')[cite: 1]
df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]

# filter out weird negative quantities or zero prices (returns/errors)[cite: 1]
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]

# fix the date column format so we can do math on it later[cite: 1]
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# calculate the actual spend per row
df['TotalAmount'] = df['Quantity'] * df['UnitPrice']

print("Clean data shape:", df.shape)
df.head()

Raw data shape: (541909, 8)
Clean data shape: (397884, 9)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalAmount
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34


In [2]:
import datetime as dt
from sklearn.preprocessing import StandardScaler

# find the latest date in the dataset to act as our "today" reference point
current_date = df['InvoiceDate'].max() + dt.timedelta(days=1)

# group everything by customer id to calculate our RFM metrics
rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (current_date - x.max()).days, # Recency
    'InvoiceNo': 'nunique',                                 # Frequency
    'TotalAmount': 'sum'                                    # Monetary
}).reset_index()

# rename the columns so we don't get confused later
rfm.columns = ['CustomerID', 'Recency', 'Frequency', 'Monetary']

print("Raw RFM Data:")
display(rfm.head())

# clustering algorithms hate unscaled data, so we standardize it (mean=0, variance=1)
scaler = StandardScaler()

# fit and transform the RFM columns (ignoring CustomerID so we don't scale an ID number)
rfm_scaled = scaler.fit_transform(rfm[['Recency', 'Frequency', 'Monetary']])

# put the scaled data back into a dataframe for the modeling step
rfm_scaled_df = pd.DataFrame(rfm_scaled, columns=['Recency', 'Frequency', 'Monetary'])

print("\nScaled RFM Data ready for clustering:")
display(rfm_scaled_df.head())

Raw RFM Data:


,CustomerID,Recency,Frequency,Monetary
0,12346.0,326,1,77183.60
1,12347.0,2,7,4310.00
2,12348.0,75,4,1797.24
3,12349.0,19,1,1757.55
4,12350.0,310,1,334.40



Scaled RFM Data ready for clustering:


,Recency,Frequency,Monetary
0,2.334574,-0.425097,8.358668
1,-0.905340,0.354417,0.250966
2,-0.175360,-0.035340,-0.028596
3,-0.735345,-0.425097,-0.033012
4,2.174578,-0.425097,-0.191347


In [3]:
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score

# Setting it to 4 clusters to match your Power BI dashboard
n_clusters = 4

print("Training models...")

# --- 1. K-Means ---
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
kmeans_labels = kmeans.fit_predict(rfm_scaled_df)

# --- 2. Hierarchical Clustering ---
hierarchical = AgglomerativeClustering(n_clusters=n_clusters)
hierarchical_labels = hierarchical.fit_predict(rfm_scaled_df)

# --- 3. DBSCAN ---
# DBSCAN is density-based and determines the number of clusters itself based on these params
dbscan = DBSCAN(eps=0.5, min_samples=10)
dbscan_labels = dbscan.fit_predict(rfm_scaled_df)

# --- Evaluate and Compare[cite: 1] ---
print("\nModel Evaluation (Silhouette Scores - closer to 1 is better):")
print(f"K-Means: {silhouette_score(rfm_scaled_df, kmeans_labels):.4f}")
print(f"Hierarchical: {silhouette_score(rfm_scaled_df, hierarchical_labels):.4f}")

# DBSCAN often struggles with RFM density and groups everything into a single cluster or noise (-1).
if len(set(dbscan_labels)) > 1:
    print(f"DBSCAN: {silhouette_score(rfm_scaled_df, dbscan_labels):.4f}")
else:
    print("DBSCAN: Failed to find distinct clusters (all data marked as noise/single cluster).")

# K-Means usually performs best for standard RFM. We will attach its labels to the main dataframe.
rfm['Cluster'] = kmeans_labels

# Let's look at the averages for each cluster so we can assign the actual Personas
cluster_summary = rfm.groupby('Cluster').agg({
    'Recency': 'mean',
    'Frequency': 'mean',
    'Monetary': 'mean',
    'CustomerID': 'count'
}).rename(columns={'CustomerID': 'CustomerCount'}).reset_index()

print("\nK-Means Cluster Averages (for Persona mapping):")
display(cluster_summary)

Training models...

Model Evaluation (Silhouette Scores - closer to 1 is better):
K-Means: 0.6162
Hierarchical: 0.6065
DBSCAN: 0.6487

K-Means Cluster Averages (for Persona mapping):


,Cluster,Recency,Frequency,Monetary,CustomerCount
0,0,43.702685,3.682711,1359.049284,3054
1,1,248.075914,1.552015,480.617480,1067
2,2,7.384615,82.538462,127338.313846,13
3,3,15.500000,22.333333,12709.090490,204


In [4]:
# Map the K-Means clusters to business personas based on the averages
persona_map = {
    2: 'Champions',
    3: 'Loyal/Regulars',
    0: 'New/Promising',
    1: 'Lost/Churned'
}

rfm['Persona'] = rfm['Cluster'].map(persona_map)

print("Persona mapping complete. Segment counts:")
print(rfm['Persona'].value_counts())

# --- 4. Predictive Classifier Training ---
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

print("\n--- Training Predictive Classifier ---")

# Define features (X) and what we want to predict (y)
X = rfm[['Recency', 'Frequency', 'Monetary']]
y = rfm['Persona']

# Split data into 80% training and 20% testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize and train the Random Forest Classifier
rf_classifier = RandomForestClassifier(n_estimators=100, random_state=42)
rf_classifier.fit(X_train, y_train)

# Test the model
y_pred = rf_classifier.predict(X_test)

# Output the results
print("\nModel Accuracy and Classification Report:")
print(classification_report(y_test, y_pred))

Persona mapping complete. Segment counts:
Persona
New/Promising     3054
Lost/Churned      1067
Loyal/Regulars     204
Champions           13
Name: count, dtype: int64

--- Training Predictive Classifier ---

Model Accuracy and Classification Report:
                precision    recall  f1-score   support

     Champions       1.00      0.50      0.67         4
  Lost/Churned       1.00      1.00      1.00       226
Loyal/Regulars       0.95      1.00      0.98        42
 New/Promising       1.00      1.00      1.00       596

      accuracy                           1.00       868
     macro avg       0.99      0.87      0.91       868
  weighted avg       1.00      1.00      1.00       868



In [5]:
import joblib

# Export the trained Random Forest model to a file
joblib.dump(rf_classifier, 'customer_segmentation_model.pkl')

print("Model successfully exported as 'customer_segmentation_model.pkl'")
print("Python pipeline complete. Ready for final zip assembly.")

Model successfully exported as 'customer_segmentation_model.pkl'
Python pipeline complete. Ready for final zip assembly.
